In [1]:
# Quick overview of available tables and their row counts
import pandas as pd
from config import engine

tables = [
    'student', 'activity_reading', 'audio_reading', 'gps_reading', 
    'bluetooth_scan', 'conversation', 'dark_period', 'phone_lock', 
    'phone_charge', 'wifi_scan', 'wifi_location', 'sms', 'call_log', 
    'app_usage', 'calendar', 'dinning', 'grade', 'deadline', 
    'piazza_usage', 'ema_definition', 'ema_response'
]

print("="*70)
print("DATABASE OVERVIEW FOR QUERY DEVELOPMENT")
print("="*70)

for table in tables:
    try:
        result = pd.read_sql(f"SELECT COUNT(*) as count FROM {table}", engine)
        print(f"{table:25s}: {result['count'][0]:>12,} rows")
    except:
        print(f"{table:25s}: Table doesn't exist")

DATABASE OVERVIEW FOR QUERY DEVELOPMENT
student                  :           49 rows
activity_reading         :   22,842,191 rows
audio_reading            :   94,995,093 rows
gps_reading              :      202,877 rows
bluetooth_scan           :    1,288,526 rows
conversation             :       79,023 rows
dark_period              :        7,269 rows
phone_lock               :        9,275 rows
phone_charge             :        3,318 rows
wifi_scan                :   19,240,217 rows
wifi_location            :    1,893,838 rows
sms                      :       92,584 rows
call_log                 :       71,801 rows
app_usage                :    1,990,510 rows
calendar                 :        5,061 rows
dinning                  :        7,617 rows
grade                    :           30 rows
deadline                 :          644 rows
piazza_usage             :           49 rows
ema_definition           :           65 rows
ema_response             :       37,358 rows


    The following are queries that are to be pasted into dbeaver, storing them here for group clarity. 

In [ ]:
-- QUERY 1: Which students sleep LESS during weeks with heavy deadlines?
-- Demonstrates: Window functions (LAG), temporal analysis, multi-table joins

WITH student_deadlines AS (
    SELECT 
        uid,
        DATE_TRUNC('week', date::timestamp) as week,
        SUM(count) as num_deadlines
    FROM deadline
    GROUP BY uid, week
),
sleep_by_week AS (
    SELECT 
        uid,
        DATE_TRUNC('week', TO_TIMESTAMP(start)) as week,
        AVG(("end" - start) / 3600.0) as avg_sleep_hours,
        COUNT(*) as sleep_sessions
    FROM dark_period
    WHERE ("end" - start) BETWEEN 3600 AND 43200
    GROUP BY uid, week
),
sleep_with_lag AS (
    SELECT 
        s.uid,
        sd.week,
        sd.num_deadlines,
        s.avg_sleep_hours,
        s.sleep_sessions,
        LAG(s.avg_sleep_hours) OVER (PARTITION BY s.uid ORDER BY s.week) as prev_week_sleep
    FROM sleep_by_week s
    JOIN student_deadlines sd ON s.uid = sd.uid AND s.week = sd.week
    WHERE sd.num_deadlines >= 3
)
SELECT 
    uid,
    TO_CHAR(week, 'YYYY-MM-DD') as week_start,
    num_deadlines,
    ROUND(avg_sleep_hours::numeric, 2) as avg_sleep_hours,
    sleep_sessions,
    ROUND(prev_week_sleep::numeric, 2) as prev_week_sleep,
    ROUND((avg_sleep_hours - prev_week_sleep)::numeric, 2) as sleep_change_hours
FROM sleep_with_lag
WHERE prev_week_sleep IS NOT NULL
ORDER BY sleep_change_hours ASC
LIMIT 15;

In [ ]:
-- QUERY 2: Which students have LOW social interaction but HIGH deadline pressure?
-- Demonstrates: Aggregation across tables, risk identification, LEFT JOIN

WITH student_conversation AS (
    SELECT 
        uid,
        SUM(duration_in_second) / 3600.0 as total_conversation_hours,
        COUNT(*) as conversation_count,
        AVG(duration_in_second / 60.0) as avg_conversation_minutes
    FROM conversation
    GROUP BY uid
),
student_deadlines AS (
    SELECT 
        uid,
        SUM(count) as total_deadlines,
        COUNT(DISTINCT date) as deadline_days
    FROM deadline
    GROUP BY uid
),
student_grades AS (
    SELECT uid, gpa
    FROM grade
)
SELECT 
    sc.uid,
    ROUND(sc.total_conversation_hours::numeric, 1) as conversation_hours,
    sc.conversation_count,
    ROUND(sc.avg_conversation_minutes::numeric, 1) as avg_conversation_min,
    sd.total_deadlines,
    sd.deadline_days,
    ROUND(sg.gpa::numeric, 2) as gpa
FROM student_conversation sc
JOIN student_deadlines sd ON sc.uid = sd.uid
LEFT JOIN student_grades sg ON sc.uid = sg.uid
WHERE sc.total_conversation_hours < 30  -- Low social interaction
  AND sd.total_deadlines >= 5           -- High academic pressure
ORDER BY sc.total_conversation_hours ASC, sd.total_deadlines DESC
LIMIT 15;

In [ ]:
-- QUERY 3: Do physically active students report lower stress and higher GPAs?
-- Demonstrates: Cross-table analysis, FILTER clause, complex aggregation

WITH student_activity AS (
    SELECT 
        uid,
        COUNT(*) as total_readings,
        SUM(CASE WHEN activity = 'walking' THEN 1 ELSE 0 END) as walking_count,
        SUM(CASE WHEN activity = 'running' THEN 1 ELSE 0 END) as running_count
    FROM activity_reading
    GROUP BY uid
),
student_stress AS (
    SELECT 
        uid,
        COUNT(*) FILTER (WHERE question_id = 'level') as stress_responses,
        AVG(CASE WHEN question_id = 'level' THEN response_value::numeric ELSE NULL END) as avg_stress
    FROM ema_response
    WHERE question_id = 'level'
    GROUP BY uid
),
student_performance AS (
    SELECT uid, gpa
    FROM grade
)
SELECT 
    sa.uid,
    sa.total_readings,
    sa.walking_count,
    sa.running_count,
    ROUND((sa.walking_count::numeric / NULLIF(sa.total_readings, 0)) * 100, 1) as pct_walking,
    COALESCE(ss.stress_responses, 0) as stress_responses,
    ROUND(ss.avg_stress::numeric, 2) as avg_stress_level,
    ROUND(sp.gpa::numeric, 2) as gpa
FROM student_activity sa
LEFT JOIN student_stress ss ON sa.uid = ss.uid
LEFT JOIN student_performance sp ON sa.uid = sp.uid
WHERE sa.total_readings > 1000  -- Active users only
ORDER BY avg_stress_level DESC NULLS LAST
LIMIT 20;

In [9]:
# Check dark_period data quality
print("DARK_PERIOD Sample:")
result = pd.read_sql("""
    SELECT 
        uid,
        start,
        "end",
        ("end" - start) / 3600.0 as duration_hours,
        TO_TIMESTAMP(start) as start_time,
        TO_TIMESTAMP("end") as end_time
    FROM dark_period
    ORDER BY uid, start
    LIMIT 20
""", engine)
print(result)

print("\n" + "="*70)
print("DARK_PERIOD Statistics:")
stats = pd.read_sql("""
    SELECT 
        COUNT(*) as total_periods,
        COUNT(DISTINCT uid) as total_students,
        AVG(("end" - start) / 3600.0) as avg_duration_hours,
        MIN(("end" - start) / 3600.0) as min_duration_hours,
        MAX(("end" - start) / 3600.0) as max_duration_hours,
        PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY ("end" - start) / 3600.0) as median_duration_hours
    FROM dark_period
    WHERE ("end" - start) BETWEEN 3600 AND 43200  -- 1-12 hours
""", engine)
print(stats)

DARK_PERIOD Sample:


DatabaseError: Execution failed on sql '
    SELECT 
        uid,
        start,
        "end",
        ("end" - start) / 3600.0 as duration_hours,
        TO_TIMESTAMP(start) as start_time,
        TO_TIMESTAMP("end") as end_time
    FROM dark_period
    ORDER BY uid, start
    LIMIT 20
': Can't reconnect until invalid transaction is rolled back.  Please rollback() fully before proceeding (Background on this error at: https://sqlalche.me/e/20/8s2b)